# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
#!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

#%pip install docling markdown-it-py
#%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Documents
import glob
from pathlib import Path

# The parser step above produces markdown files in data_dir
list_md_files = sorted(glob.glob(f"{data_dir}/*.md"))

if not list_md_files:
    raise FileNotFoundError(f"No markdown files found in {data_dir}. Run Step 1 first.")

print(f"Found {len(list_md_files)} markdown files")
for fp in list_md_files:
    print(f"  - {Path(fp).name}")


In [ ]:
# Step 4A: Setup, Chunking and Query Helpers

from markdown_it import MarkdownIt
from typing import List, Optional
from pathlib import Path
from dotenv import load_dotenv
from sdg_hub import Flow, FlowRegistry
from sdg_hub.core.utils.translation import translate_flow
import datasets
import pandas as pd
import os
import re


load_dotenv()


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """Split markdown text into word chunks while preserving overlap."""
    md = MarkdownIt()
    tokens = md.parse(text)

    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        for word in block.split():
            current_words.append(word)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def set_model_config(flow_object):
    """Configure flow model settings from .env, matching knowledge_generation logic."""
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
    print(f"Using model provider: {model_provider}")

    if model_provider == "hosted_vllm":
        flow_object.set_model_config(
            model=os.getenv("VLLM_MODEL", "hosted_vllm/meta-llama/Llama-3.3-70B-Instruct"),
            api_base=os.getenv("VLLM_API_BASE", "http://localhost:8000/v1"),
            api_key=os.getenv("VLLM_API_KEY", "EMPTY"),
            enable_reasoning=os.getenv("ENABLE_REASONING", "false").lower() in ("1", "true", "yes"),
        )
    elif model_provider == "openai":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    elif model_provider == "openrouter":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
            api_base="https://openrouter.ai/api/v1",
        )
    elif model_provider == "ollama":
        flow_object.set_model_config(
            model=os.getenv("OLLAMA_MODEL", "ollama/gemma2"),
            api_base=os.getenv("OLLAMA_API_BASE", "http://localhost:11434"),
        )
    elif model_provider == "maas":
        flow_object.set_model_config(
            model=os.getenv("MAAS_MODEL"),
            api_base=os.getenv("MAAS_API_BASE"),
            api_key=os.getenv("MAAS_API_KEY"),
        )
    else:
        raise ValueError(f"Unsupported MODEL_PROVIDER: {model_provider}")

    return flow_object


def truncate_words(text: str, max_words: int) -> str:
    words = str(text or "").split()
    return str(text or "") if len(words) <= max_words else " ".join(words[:max_words])


def compact_document_for_prompt(text: str, max_words: int = 4500) -> str:
    words = str(text or "").split()
    if len(words) <= max_words:
        return str(text or "")

    head_size = int(max_words * 0.75)
    tail_size = max_words - head_size
    head = " ".join(words[:head_size])
    tail = " ".join(words[-tail_size:])
    return f"{head}\n\n[... CONTENT TRUNCATED ...]\n\n{tail}"


def normalize_query(q: str) -> str:
    cleaned = str(q or "").replace("<Insert question here>", "").strip()
    return re.sub(r"\s+", " ", cleaned)


def fallback_queries(document_outline: str) -> List[str]:
    return [
        f"Quais são as ideias principais apresentadas em {document_outline}?",
        f"Como os pontos centrais de {document_outline} podem ser aplicados na prática?",
        f"Quais limitações ou trade-offs são discutidos em {document_outline}?",
    ]


def is_likely_non_ptbr_question(text: str) -> bool:
    question = str(text or "").strip().lower()
    if not question:
        return True

    english_starts = (
        "what ", "how ", "which ", "why ", "when ", "where ", "who ",
        "is ", "are ", "can ", "does ", "do ",
    )
    if question.startswith(english_starts):
        return True

    english_tokens = {
        "what", "how", "which", "why", "when", "where", "who", "the", "and", "for", "with", "from", "about"
    }
    portuguese_tokens = {
        "como", "qual", "quais", "quando", "onde", "por", "que", "de", "em", "feriado", "emenda", "documento"
    }
    tokens = re.findall(r"[a-záéíóúàâêôãõç]+", question)
    en_hits = sum(1 for token in tokens if token in english_tokens)
    pt_hits = sum(1 for token in tokens if token in portuguese_tokens)
    return en_hits >= 2 and pt_hits == 0


def is_low_quality_query(text: str) -> bool:
    query = normalize_query(text)
    if not query:
        return True

    lower = query.lower()
    blocked_patterns = (
        "...",
        "…",
        "<insert question here>",
        "[question]",
        "[end]",
        "<insira a pergunta aqui>",
    )
    if any(pattern in lower for pattern in blocked_patterns):
        return True

    # Avoid fragments and malformed prompts.
    words = query.split()
    if len(words) < 6:
        return True
    if "?" not in query:
        return True

    return False


def enforce_ptbr_queries(queries: List[str], document_outline: str) -> List[str]:
    defaults = fallback_queries(document_outline)
    normalized = [normalize_query(q) for q in (queries or []) if normalize_query(q)]

    result = []
    for i, query in enumerate(normalized[:3]):
        if is_low_quality_query(query) or is_likely_non_ptbr_question(query):
            result.append(defaults[i])
        else:
            result.append(query)

    while len(result) < 3:
        result.append(defaults[len(result)])

    return result[:3]


In [ ]:
# Step 4B: Framework-First Execution Notes

# O parsing e extração principais são feitos no flow oficial (flow.yaml).
# Esta célula é mantida apenas para separar etapas e facilitar leitura/manutenção.


In [ ]:
# Step 4C: Discover and Load ICL Bootstrap Flow

# Load official flow and apply model config/fallbacks
flow_id = "brisk-cinder-icl-bootstrap"
flow_name = "ICL Bootstrap Metadata and Question Seeding Flow"


def register_local_flows_path() -> Optional[str]:
    """Register local source-tree flows so notebooks work without package reinstall."""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        local_flows = base / "src" / "sdg_hub" / "flows"
        if local_flows.exists():
            FlowRegistry.register_search_path(str(local_flows))
            # FlowRegistry caches discovered entries; force refresh after adding a path.
            if hasattr(FlowRegistry, "_discover_flows"):
                FlowRegistry._discover_flows(force_refresh=True)
            return str(local_flows)
    return None


# Read multilingual settings from .env (optional)
sdg_lang = os.getenv("SDG_LANG", "").strip()
sdg_lang_code = os.getenv("SDG_LANG_CODE", "").strip()
if sdg_lang and not sdg_lang_code:
    raise ValueError("SDG_LANG is set but SDG_LANG_CODE is missing. Both are required.")
if sdg_lang_code and not sdg_lang:
    raise ValueError("SDG_LANG_CODE is set but SDG_LANG is missing. Both are required.")

# Register local source-tree flows first
local_flows_path = register_local_flows_path()
if local_flows_path:
    print(f"Registered local flows path: {local_flows_path}")

# If a custom translated flows directory exists, register it for discovery
translated_flows_dir = os.getenv("TRANSLATED_FLOWS_DIR", "").strip()
if translated_flows_dir and os.path.isdir(translated_flows_dir):
    FlowRegistry.register_search_path(translated_flows_dir)
    if hasattr(FlowRegistry, "_discover_flows"):
        FlowRegistry._discover_flows(force_refresh=True)


def ensure_translated_flow(base_flow_name: str) -> str:
    """Return translated flow name when multilingual mode is enabled."""
    if not sdg_lang:
        return base_flow_name

    translated_name = f"{base_flow_name} ({sdg_lang})"
    if FlowRegistry.get_flow_path(translated_name) is not None:
        print(f"Found translated flow: {translated_name}")
        return translated_name

    # Custom local flow already uses localized prompts (pt-BR), so translation is optional.
    if base_flow_name == flow_name:
        print(
            f"Translated variant not found for '{base_flow_name}'. "
            "Using base flow because prompts are already localized."
        )
        return base_flow_name

    print(f"Translating flow on-demand: {base_flow_name} -> {sdg_lang}")

    source_path = FlowRegistry.get_flow_path(base_flow_name)
    if source_path is None:
        source_path = FlowRegistry.get_flow_path(flow_id)

    output_dir = None
    if translated_flows_dir and source_path is not None:
        flow_subdir = Path(source_path).parent.name
        output_dir = str(Path(translated_flows_dir) / f"{flow_subdir}_{sdg_lang_code}")

    translate_flow(
        flow=base_flow_name,
        lang=sdg_lang,
        lang_code=sdg_lang_code,
        translator_model=os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o"),
        verifier_model=os.getenv(
            "VERIFIER_MODEL", os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o")
        ),
        output_dir=output_dir,
        translator_api_key=os.getenv("TRANSLATOR_API_KEY"),
        translator_api_base=os.getenv("TRANSLATOR_API_BASE"),
        verifier_api_key=os.getenv("VERIFIER_API_KEY"),
        verifier_api_base=os.getenv("VERIFIER_API_BASE"),
        register=True,
    )

    return translated_name


resolved_flow_name = ensure_translated_flow(flow_name)
if sdg_lang:
    print(f"Multilingual mode: {sdg_lang} ({sdg_lang_code})")
else:
    print("Language: English (default)")

# Prefer lookup by translated name in multilingual mode.
# In default mode keep id-first lookup for stability.
if sdg_lang:
    flow_path = FlowRegistry.get_flow_path(resolved_flow_name)
    if flow_path is None:
        flow_path = FlowRegistry.get_flow_path(flow_id) or FlowRegistry.get_flow_path(flow_name)
        if flow_path is not None:
            print(
                "Translated flow not available in registry; "
                f"falling back to base flow: {flow_name}"
            )
else:
    flow_path = FlowRegistry.get_flow_path(flow_id) or FlowRegistry.get_flow_path(resolved_flow_name)

if flow_path is None:
    available = []
    try:
        available = FlowRegistry.list_flows()
    except Exception:
        pass
    available_preview = [f"{f.get('id')} ({f.get('name')})" for f in available[:10]]
    raise FileNotFoundError(
        "Flow not found in registry: "
        f"{flow_id} / {resolved_flow_name}. "
        f"Available flows (first 10): {available_preview}"
    )

print(f"Using flow id: {flow_id}")
print(f"Using flow name: {resolved_flow_name}")
print(f"Flow path: {flow_path}")

icl_bootstrap_flow = Flow.from_yaml(flow_path)
icl_bootstrap_flow = set_model_config(icl_bootstrap_flow)


In [ ]:
# Step 4D: Run Per-Document ICL Generation and Save seed_data.jsonl

# Process every markdown document and replicate per-doc ICL across its chunks
seed_rows = []
processed_docs = 0
skipped_docs = 0

for md_fp in list_md_files:
    doc_name = Path(md_fp).name
    print(f"\nProcessing: {doc_name}")

    try:
        with open(md_fp, "r", encoding="utf-8") as f:
            doc_text = f.read()

        chunks = chunk_markdown(doc_text, max_tokens=5000, overlap=1000)
        if not chunks:
            print("  - Skipping: no chunks generated")
            skipped_docs += 1
            continue

        flow_input = pd.DataFrame(
            [
                {
                    "document": compact_document_for_prompt(doc_text, max_words=4500),
                    "document_title": Path(md_fp).stem,
                }
            ]
        )

        flow_output = icl_bootstrap_flow.generate(flow_input, max_concurrency=1)
        if flow_output.empty:
            print("  - Skipping: flow output is empty")
            skipped_docs += 1
            continue

        first_row = flow_output.iloc[0].to_dict()
        document_outline = str(first_row.get("document_outline") or "").strip() or Path(md_fp).stem.replace("_", " ")
        domain = str(first_row.get("domain") or "").strip() or "articles/essays"
        icl_document = truncate_words(str(first_row.get("icl_document") or "").strip() or doc_text, 1600)

        seed_qs = [
            normalize_query(first_row.get("seed_query_1", "")),
            normalize_query(first_row.get("seed_query_2", "")),
            normalize_query(first_row.get("seed_query_3", "")),
        ]
        seed_defaults = fallback_queries(document_outline)
        for i in range(3):
            if not seed_qs[i]:
                seed_qs[i] = seed_defaults[i]

        generated_questions = []
        if "question" in flow_output.columns:
            generated_questions = [normalize_query(q) for q in flow_output["question"].tolist()]
            generated_questions = [q for q in generated_questions if q]

        final_questions = []
        for q in generated_questions + seed_qs:
            if q and q not in final_questions:
                final_questions.append(q)

        while len(final_questions) < 3:
            final_questions.append(seed_defaults[len(final_questions)])
        final_questions = final_questions[:3]
        final_questions = enforce_ptbr_queries(final_questions, document_outline)

        for chunk in chunks:
            seed_rows.append(
                {
                    "document": chunk,
                    "document_outline": document_outline,
                    "icl_document": icl_document,
                    "icl_query_1": final_questions[0],
                    "icl_query_2": final_questions[1],
                    "icl_query_3": final_questions[2],
                    "domain": domain,
                }
            )

        print(f"  - Chunks: {len(chunks)}")
        print(f"  - Outline: {document_outline}")
        print(f"  - Domain: {domain}")
        processed_docs += 1

    except Exception as exc:
        print(f"  - Skipping due to error: {exc}")
        skipped_docs += 1

if not seed_rows:
    raise ValueError("No seed rows generated. Check data_dir and model configuration.")

# Build and validate final dataset schema
required_cols = [
    "document",
    "document_outline",
    "icl_document",
    "icl_query_1",
    "icl_query_2",
    "icl_query_3",
    "domain",
]

seed_data = datasets.Dataset.from_list(seed_rows).select_columns(required_cols)

for col in ["document_outline", "icl_document", "icl_query_1", "icl_query_2", "icl_query_3", "domain"]:
    empty_count = sum(1 for v in seed_data[col] if not str(v).strip())
    if empty_count > 0:
        raise ValueError(f"Column {col} has {empty_count} empty values after ICL generation")

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)

print("\n" + "=" * 60)
print("ICL AUTOMATIC GENERATION SUMMARY")
print("=" * 60)
print(f"Processed documents: {processed_docs}")
print(f"Skipped documents:   {skipped_docs}")
print(f"Total output rows:   {len(seed_data)}")
print("Saved: seed_data.jsonl")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook